### Installing Libraries

In [ ]:
# BitsAndBytes is a library for 8-bit optimizers and quantization of large language models. 
# It allows you to train and fine-tune large language models using 8-bit precision, which can significantly reduce memory usage and speed up training.
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

### Connecting to Tesla T4 GPU

In [ ]:
# GPU Information
gpu_info = !nvidia-smi
gpu_info

In [ ]:
# Check the T4 status
gpu_info = '\n'.join(gpu_info)
if 'Tesla T4' in gpu_info:
    print("Success - Connected to a T4")
else:
    print("NOT CONNECTED TO A T4")

### Connecting to HuggingFace

In [ ]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Enter HuggingFace Token: ")
login(token=hf_token, add_to_git_credential=True)

### Accessing Models

In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"
FALCON = "tiiuae/falcon-7b-instruct"
GEMMA = "google/gemma-3-270m-it"

In [ ]:
messages = [
    {"role": "user", "content": "Tell users regarding HuggingFace Transformers libraries."}
  ]

### Quantization Config

In [ ]:
# This allows to load the model in 8-bit precision, which can significantly reduce memory usage and speed up inference.

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load the model in 4-bit precision
    bnb_4bit_use_double_quant=True,  # Use double quantization for better accuracy
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for computation
    bnb_4bit_quant_type="nf4",  # Use NormalFloat4 quantization
)

quant_config

### Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(FALCON)
tokenizer.pad_token = tokenizer.eos_token # Set the pad token to the end-of-sequence token, this is important for proper padding during generation.
input_ids = tokenizer.apply_chat_template(
    messages, 
    return_tensors="pt",  # Return the input as PyTorch tensors.
    padding=True, # Pad the input sequences to the same length, which is necessary for batch processing.
    truncation=True, # Truncate the input sequences to fit within the model's maximum input length, preventing errors during generation.
    ).to("cuda")

In [ ]:
input_ids

### Model

In [ ]:
# Load the model with the specified quantization configuration and automatically map it to the available device.
# AutoModelForCausalLM is useful for loading causal language models, which helps in efficiently utilizing GPU memory while maintaining a good balance between performance and accuracy.
model = AutoModelForCausalLM.from_pretrained(FALCON, quantization_config=quant_config, device_map="auto")
model

Now take a look at the falcon-7b-instruct's layers of the Neural Network:

- The model is made up of multiple layers.
- There is an embedding layer (word_embeddings) that converts input tokens into 4,544-dimensional vectors.
- There are 32 stacked decoder layers. Each decoder layer contains:
    - (a) Self-attention layers
    - (b) Multi-Layer Perceptron (MLP) layers
    - (c) Layer normalization layers
- At the end of the model, there is an LM head (lm_head) that maps the final hidden states back to the vocabulary size (65,024 tokens) to produce output logits.

Also note that many of the linear layers are shown as Linear4bit. This indicates that the model has been quantized to 4-bit precision, which reduces memory usage and speeds up inference while slightly reducing numerical precision.

In [ ]:
#  Embedding(65024, 4544) means that the embedding layer has 65024 tokens in the vocabulary, and each token is represented by a 4544-dimensional vector. 
#  The embedding dimension is chosen to capture the semantic information of the tokens effectively.

In [ ]:
# FalconRotaryEmbedding helps the model understand the position of tokens in a sequence, which is crucial for tasks like language modeling and text generation. 
# The "rotary" aspect refers to a specific way of encoding positional information that can improve the model's ability to capture long-range dependencies in the input data.

In [ ]:
# Encoders are responsible for processing the input sequence and creating a representation that the decoder can use to generate the output.
# Decoders are responsible for generating the output sequence based on the input and the model's learned representations.

In [ ]:
memory = model.get_memory_footprint()
print(f"Model Memory Footprint: {memory / (1024 ** 3):.2f} GB")

In [ ]:
#  attention mask is a binary tensor that indicates which tokens in the input sequence should be attended to and which should be ignored. 
#  It helps the model focus on relevant parts of the input while generating the output, improving the quality of the generated text.
attention_mask = torch.ones_like(input_ids, dtype=torch.long).to("cuda")
attention_mask

In [ ]:
output = model.generate(
    input_ids=input_ids, # The input_ids are the tokenized representation of the input text, which the model uses to generate the output.
    attention_mask=attention_mask, # The attention mask is a binary tensor that indicates which tokens in the input sequence should be attended to and which should be ignored.
    pad_token_id=tokenizer.pad_token_id, # The pad_token_id is important for ensuring that all input sequences have the same length during batch processing.
    max_new_tokens=80 # The max_new_tokens parameter specifies the maximum number of new tokens that the model should generate in response to the input.
    )
output[0]

In [ ]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Stream

In [ ]:
def generate(model, message, quant=True, max_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token 

    input_ids = tokenizer.apply_chat_template(
        message, 
        return_tensors="pt", 
        padding=True, 
        truncation=True).to("cuda")
    
    attention_mask = torch.ones_like(input_ids, dtype=torch.long).to("cuda")
    streamer = TextStreamer(tokenizer)

    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
    output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_tokens, streamer=streamer)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
model = "Qwen/Qwen2.5-Coder-7B-Instruct"
messages = [
    {"role": "user", "content": "Tell users what is computers and how it works?"}
  ]

In [ ]:
generate(model=model, message=messages)